# Synthetic Calibration Data Generator

Simulates flume carriage runs for a given propeller and set of target velocities.

Each file reproduces a realistic run:
- Short acceleration ramp at the start
- Constant-velocity plateau (the calibration window)
- Short deceleration ramp at the end
- Gaussian noise on the rev/s signal throughout

Output format matches the ESP32 recording files so the calibration notebook can read them directly.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta

## Configuration

In [ ]:
# ── Propeller calibration curve (true physical relationship) ──────────────────
# For IMP1: v_true = a + b * rev_s
# We simulate the SENSOR output (rev/s) from a known true velocity.
# Inverse: rev_s = (v_true - a) / b
PROP_ID  = "IMP1"
CAL_A    = 0.03   # m/s intercept  (small start-up friction offset)
CAL_B    = 0.065  # m/s per rev/s  (linear factor)

def true_velocity_to_revs(v_true):
    """Convert true water velocity [m/s] to expected rev/s for IMP1."""
    return max(0.0, (v_true - CAL_A) / CAL_B)

# ── Run parameters ────────────────────────────────────────────────────────────
TARGET_VELOCITIES = [0.3, 0.6, 0.9, 1.2, 1.5]  # m/s

DT            = 0.5    # s — reporting interval (matches app setting)
RAMP_DURATION = 5.0    # s — acceleration / deceleration ramp length
PLATEAU_DURATION = 20.0  # s — constant-velocity window
NOISE_STD     = 0.3    # rev/s — Gaussian noise std on the sensor signal

OUTPUT_DIR = Path("synthetic_data")
OUTPUT_DIR.mkdir(exist_ok=True)

## Generator

In [ ]:
def generate_run(v_target, dt, ramp_s, plateau_s, noise_std, prop_id, seed=None):
    """
    Returns a DataFrame with columns [timestamp, elapsed_s, value]
    where value is the simulated raw sensor output in rev/s.
    """
    rng = np.random.default_rng(seed)
    total_s = ramp_s + plateau_s + ramp_s
    times   = np.arange(0, total_s + dt, dt)

    revs_target = true_velocity_to_revs(v_target)

    true_revs = np.zeros(len(times))
    for i, t in enumerate(times):
        if t <= ramp_s:
            # smooth cosine acceleration
            true_revs[i] = revs_target * 0.5 * (1 - np.cos(np.pi * t / ramp_s))
        elif t <= ramp_s + plateau_s:
            true_revs[i] = revs_target
        else:
            t_dec = t - ramp_s - plateau_s
            true_revs[i] = revs_target * 0.5 * (1 + np.cos(np.pi * t_dec / ramp_s))

    # Add Gaussian noise; clip to >= 0 (sensor can't report negative)
    noisy_revs = np.clip(true_revs + rng.normal(0, noise_std, len(times)), 0, None)

    t0 = datetime(2026, 4, 1, 10, 0, 0)
    timestamps = [t0 + timedelta(seconds=float(t)) for t in times]

    df = pd.DataFrame({
        "timestamp": [ts.strftime("%Y-%m-%d %H:%M:%S") for ts in timestamps],
        "elapsed_s": np.round(times, 3),
        "value":     np.round(noisy_revs, 3),
    })
    return df


def save_run(df, v_target, prop_id, dt, output_dir):
    v_str   = f"{v_target:.1f}".replace(".", "_")
    fname   = f"Calibration_{prop_id}_{v_str}_mps.txt"
    fpath   = output_dir / fname

    with open(fpath, "w", encoding="utf-8") as f:
        f.write(f"# OTT Velocimeter ESP32 — Calibration Run (SYNTHETIC)\n")
        f.write(f"# Propeller: {prop_id}\n")
        f.write(f"# Target velocity: {v_target} m/s\n")
        f.write(f"# Interval (s): {dt}\n")
        f.write(f"# Mode: Count / DeltaT\n")
        f.write(f"# Calibration: UNCALIBRATED (values in rev/s)\n")
        f.write(f"# ---\n")
        f.write("timestamp,elapsed_s,value\n")
        df.to_csv(f, index=False, header=False)

    print(f"Saved: {fpath}")
    return fpath

## Generate files

In [ ]:
generated = []
for i, v in enumerate(TARGET_VELOCITIES):
    df   = generate_run(v, DT, RAMP_DURATION, PLATEAU_DURATION, NOISE_STD, PROP_ID, seed=i)
    path = save_run(df, v, PROP_ID, DT, OUTPUT_DIR)
    generated.append(path)

print(f"\nGenerated {len(generated)} synthetic calibration files in '{OUTPUT_DIR}/'")

## Quick visualisation

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
for path in generated:
    df = pd.read_csv(path, comment="#")
    df.columns = ["timestamp", "elapsed_s", "value"]
    ax.plot(df["elapsed_s"], df["value"], label=path.stem)

ax.set_xlabel("Elapsed time [s]")
ax.set_ylabel("Sensor output [rev/s]")
ax.set_title(f"Synthetic calibration runs — {PROP_ID}")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()